## Context Precision
The ContextPrecision metric evaluates whether retrieved contexts are useful for answering a question by comparing each context against a reference answer. Use this when you have a reference answer available.

| Field                | Purpose                                    |
| -------------------- | ------------------------------------------ |
| `reference`          | Ground-truth answer (what should be known) |
| `response`           | Model-generated answer                     |
| `retrieved_contexts` | Retrieved chunks from the vector store     |
| `user_input`         | Original question                          |


In [1]:
import sys
print(sys.executable)

/home/sathw/.pyenv/versions/3.11.9/bin/python


In [ ]:
from pathlib import Path
# from langchain.document_loaders import PyPDFLoader
from langchain_community.document_loaders import PyPDFLoader
# from langchain_community.document_loaders import TextLoader, PyPDFLoader

documents = []

data_dir = Path("./data")

for file in data_dir.iterdir():

    if file.suffix.lower() == ".txt":
        print(f"Loading TXT: {file}")
        documents.extend(TextLoader(str(file)).load())

    elif file.suffix.lower() == ".pdf":
        print(f"Loading PDF: {file}")
        documents.extend(PyPDFLoader(str(file)).load())

print(f"Total documents loaded: {len(documents)}")


ImportError: cannot import name 'BaseBlobParser' from 'langchain_core.document_loaders' (/home/sathw/.pyenv/versions/3.11.9/lib/python3.11/site-packages/langchain_core/document_loaders/__init__.py)

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
chunks=splitter.split_documents(documents)
print("Splitter created Successfully")
print(f"length of chunks:{len(chunks)}")


NameError: name 'documents' is not defined

In [1]:
from sentence_transformers import SentenceTransformer

model= SentenceTransformer("all-MiniLM-L6-v2")
texts=[chunk.page_content for chunk in chunks]

embeddings=model.encode(
            texts,
            convert_to_numpy=True
        )
print(embeddings.shape)

/home/sathw/bootcamp-project/ragas_clean/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 299.76it/s]


NameError: name 'chunks' is not defined

In [3]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

/tmp/ipykernel_3886/211093956.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
/tmp/ipykernel_3886/211093956.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
/tmp/ipykernel_3886/211093956.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import faithfulness,

In [4]:
data = [
    {
        "question":    "What is the refund policy?",
        "answer":      "You can get a full refund within 30 days of purchase.",
        "contexts":    ["Customers may request a full refund within 30 days of purchase."],
        "ground_truth":"Full refund is available within 30 days.",
    },
    {
        "question":    "What was the Q3 revenue?",
        "answer":      "Q3 revenue was $4.2M, up 18%.",          # hallucinated — not in context
        "contexts":    ["Q3 showed strong growth across all segments."],
        "ground_truth":"Q3 revenue figures were not publicly disclosed.",
    },
    {
        "question":    "How do I reset my password?",
        "answer":      "Go to Settings → Security → Reset Password and follow the steps.",
        "contexts":    ["To reset your password: Settings → Security → Reset Password."],
        "ground_truth":"Navigate to Settings > Security > Reset Password.",
    },
]

In [ ]:
# ----------------------------
# Bedrock LLM
# ----------------------------
llm = ChatBedrock(
    model_id="amazon.nova-micro-v1:0",
    region_name="us-east-1",
)

# ----------------------------
# Embeddings
# ----------------------------
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# ----------------------------
# Dataset
# ----------------------------
data = [
    {
        "question": "What is the refund policy?",
        "answer": "You can get a full refund within 30 days of purchase.",
        "contexts": [
            "Customers may request a full refund within 30 days of purchase."
        ],
        "ground_truth": "Full refund is available within 30 days."
    },
    {
        "question": "What was the Q3 revenue?",
        "answer": "Q3 revenue was $4.2M, up 18%.",
        "contexts": [
            "Q3 showed strong growth across all segments."
        ],
        "ground_truth": "Q3 revenue figures were not publicly disclosed."
    },
    {
        "question": "How do I reset my password?",
        "answer": "Go to Settings → Security → Reset Password and follow the steps.",
        "contexts": [
            "To reset your password: Settings → Security → Reset Password."
        ],
        "ground_truth": "Navigate to Settings > Security > Reset Password."
    },
]

dataset = Dataset.from_list(data)

# ----------------------------
# Evaluation
# ----------------------------
results = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ],
    llm=llm,
    embeddings=embeddings,
)

df = results.to_pandas()

print(df)

print("\nAverage Scores:")
print(df[
    [
        "faithfulness",
        "answer_relevancy",
        "context_precision",
        "context_recall",
    ]
].mean())

df.to_csv("scores.csv", index=False)

In [ ]:
# RAG Eval Lab — Simple Version
# pip install ragas langchain-openai datasets pandas
# export OPENAI_API_KEY=sk-...

from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

# ── 1. YOUR DATA ───────────────────────────────────────────────────────────────
# Each entry = one query your RAG system answered
data = [
    {
        "question":    "What is the refund policy?",
        "answer":      "You can get a full refund within 30 days of purchase.",
        "contexts":    ["Customers may request a full refund within 30 days of purchase."],
        "ground_truth":"Full refund is available within 30 days.",
    },
    {
        "question":    "What was the Q3 revenue?",
        "answer":      "Q3 revenue was $4.2M, up 18%.",          # hallucinated — not in context
        "contexts":    ["Q3 showed strong growth across all segments."],
        "ground_truth":"Q3 revenue figures were not publicly disclosed.",
    },
    {
        "question":    "How do I reset my password?",
        "answer":      "Go to Settings → Security → Reset Password and follow the steps.",
        "contexts":    ["To reset your password: Settings → Security → Reset Password."],
        "ground_truth":"Navigate to Settings > Security > Reset Password.",
    },
]

# ── 2. RUN RAGAS ───────────────────────────────────────────────────────────────
dataset = Dataset.from_list(data)

results = evaluate(
    dataset = dataset,
    metrics = [faithfulness, answer_relevancy, context_precision, context_recall],
)

# ── 3. PRINT SCORES ────────────────────────────────────────────────────────────
df = results.to_pandas()

print("\n── Per-query scores ──")
for _, row in df.iterrows():
    print(f"\nQ: {row['question']}")
    print(f"  Faithfulness     : {row['faithfulness']:.2f}")
    print(f"  Answer relevancy : {row['answer_relevancy']:.2f}")
    print(f"  Context precision: {row['context_precision']:.2f}")
    print(f"  Context recall   : {row['context_recall']:.2f}")

print("\n── Averages ──")
for col in ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]:
    print(f"  {col:<22}: {df[col].mean():.2f}")

df.to_csv("scores.csv", index=False)
print("\nSaved → scores.csv")

In [ ]:
{ "question": "What is the refund policy for annual subscriptions?", 
  "answer": "Annual subscriptions are refundable within 30 days of purchase.",
  "contexts": ["Customers may request a full refund within 30 days of the initial purchase date for annual plans."], 
  "ground_truth": "Full refund available within 30 days for annual subscriptions."
}
{"question": "How do I reset my password?", 
 "answer": "Go to Settings → Security → Reset Password and follow the email instructions.", 
 "contexts": ["To reset your password, navigate to Settings, then Security, then click Reset Password. An email will be sent."], 
 "ground_truth": "Reset password via Settings > Security > Reset Password, then follow the email link."}

{"question": "What is the API rate limit?", 
 "answer": "The API allows 1000 requests per minute per API key.", 
 "contexts": ["Rate limits: 1000 requests/minute per API key. Exceeding this returns a 429 error."], 
 "ground_truth": "1000 requests per minute per API key."
 }
{"question": "How does the OAuth 2.0 authentication flow work?", 
 "answer": "The client redirects the user to /auth/authorize, receives a code, then exchanges it at /auth/token for an access token.", 
 "contexts": ["OAuth 2.0 Authorization Code flow: redirect to /auth/authorize, receive code, exchange at /auth/token.", "Include Authorization: Bearer <token> in all requests."], 
 "ground_truth": "Authorization Code flow: redirect → code → token exchange → Bearer header."
 }
{"question": "What changed in the v2 API migration?", 
 "answer": "Pagination changed from offset/limit to cursor-based. The /users endpoint was renamed to /accounts.", 
 "contexts": ["v2 breaking changes: pagination is now cursor-based.", "The /users endpoint is deprecated; use /accounts instead."], 
 "ground_truth": "Cursor pagination replaces offset/limit; /users renamed to /accounts."
 }
{"question": "What was the Q3 revenue?", 
 "answer": "Q3 revenue was $4.2M, up 18% quarter-over-quarter.", "contexts": ["Q3 showed strong growth across all business segments."], "ground_truth": "Q3 revenue figures were not publicly disclosed."}

{"question": "How do I set up SSO with SAML?", 
 "answer": "Upload your IdP metadata XML in Settings → SSO, then map the email attribute.", "contexts": ["SAML SSO setup: go to Settings → SSO, upload your IdP metadata XML file, and map the email attribute to user identity."], "ground_truth": "Upload IdP metadata XML in Settings > SSO and map the email attribute."}

{"question": "What is the maximum file upload size?", 
 "answer": "The maximum file upload size is 50MB per file.", "contexts": ["File uploads are limited to 50MB per file. Files exceeding this limit will be rejected with a 413 error."], "ground_truth": "50MB per file maximum upload size."}

{"question": "How does semantic chunking differ from fixed-size chunking?", 
 "answer": "Semantic chunking splits text at meaningful boundaries like paragraphs or sentences, while fixed-size chunking splits every N characters regardless of content.", "contexts": ["Semantic chunking uses NLP to split at sentence or paragraph boundaries.", "Fixed-size chunking splits every N tokens regardless of content structure."], "ground_truth": "Semantic chunking respects content boundaries; fixed-size splits every N tokens blindly."}

{"question": "What is HyDE in retrieval?", 
 "answer": "HyDE stands for Hypothetical Document Embeddings. It generates a fake answer to the query and uses its embedding for retrieval instead of the raw query.", "contexts": ["HyDE (Hypothetical Document Embeddings): generate a hypothetical answer, embed it, use that embedding for retrieval to improve semantic match."], "ground_truth": "HyDE generates a hypothetical answer and uses its embedding for retrieval."}

{"question": "How do I troubleshoot 429 errors?", 
 "answer": "A 429 error means you have exceeded the rate limit. Implement exponential backoff and retry logic.", "contexts": ["429 Too Many Requests: rate limit exceeded. Implement exponential backoff with jitter. Rate resets every 60 seconds."], "ground_truth": "429 means rate limit exceeded. Use exponential backoff; limit resets every 60 seconds."}

{"question": "What embedding models are supported?", 
 "answer": "The platform supports OpenAI text-embedding-3-small, text-embedding-3-large, and Cohere embed-english-v3.0.", "contexts": ["Supported embedding models: OpenAI text-embedding-3-small, text-embedding-3-large, and Cohere embed-english-v3.0."], "ground_truth": "Supported: OpenAI text-embedding-3-small, text-embedding-3-large, Cohere embed-english-v3.0."}

{"question": "How does MMR filtering work in retrieval?", 
 "answer": "MMR (Maximal Marginal Relevance) selects chunks that are relevant to the query but dissimilar to already selected chunks, reducing redundancy.", "contexts": ["MMR balances relevance and diversity. Each new chunk is selected to maximise relevance to the query while minimising similarity to already selected chunks."], "ground_truth": "MMR selects relevant but diverse chunks by penalising similarity to already chosen results."}

{"question": "What is the difference between context precision and context recall?", 
 "answer": "Context precision measures how many retrieved chunks are relevant. Context recall measures whether all necessary information was retrieved.", "contexts": ["Context precision: relevant retrieved / total retrieved.", "Context recall: ground-truth covered by context / total ground-truth needed."], "ground_truth": "Precision = retrieved relevance ratio; Recall = coverage of needed information."}

{"question": "How do I compare two products in the knowledge base?", 
 "answer": "Product A supports cloud deployment while Product B is on-premise only.", "contexts": ["Product A: cloud-native deployment on AWS and GCP."], "ground_truth": "Product A is cloud-native; Product B is on-premise with optional hybrid support."}

{"question": "What is cross-encoder re-ranking?", 
 "answer": "Cross-encoder re-ranking takes the query and each retrieved chunk together as input to a model that scores their relevance jointly, more accurately than bi-encoder retrieval.", "contexts": ["Cross-encoders score query-document pairs jointly, giving higher accuracy than bi-encoders at the cost of latency."], "ground_truth": "Cross-encoders jointly score query and document pairs for more accurate relevance ranking."}

{"question": "What is the context window limit for the API?", 
 "answer": "The API supports a maximum context window of 128,000 tokens.", "contexts": ["Maximum context window: 128,000 tokens. Requests exceeding this will return a 400 error."], "ground_truth": "128,000 token maximum context window."}

{"question": "How do I enable multi-factor authentication?", 
 "answer": "Go to Settings → Security → Two-Factor Authentication and scan the QR code with your authenticator app.", "contexts": ["Enable MFA: Settings → Security → Two-Factor Authentication. Scan the QR code using Google Authenticator or Authy."], "ground_truth": "Enable MFA via Settings > Security > Two-Factor Authentication, scan QR code."}

{"question": "What is BM25 and when should I use it?", 
 "answer": "BM25 is a keyword-based ranking algorithm. Use it when queries contain exact terms or product codes that semantic search may miss.", "contexts": ["BM25 ranks documents by keyword frequency. Best for exact-match queries like product IDs, codes, or specific terminology."], "ground_truth": "BM25 is keyword-based; best for exact-match or terminology-heavy queries."}

{"question": "What are the pricing tiers?", 
 "answer": "There are three tiers: Starter at $29/month, Pro at $99/month, and Enterprise with custom pricing.", "contexts": ["Pricing: Starter $29/month (up to 5 users), Pro $99/month (up to 50 users), Enterprise — contact sales."], "ground_truth": "Starter $29/mo, Pro $99/mo, Enterprise custom pricing."}

{"question": "How does query decomposition work?", 
 "answer": "Query decomposition breaks a complex question into simpler sub-queries, retrieves context for each, then combines the results before generation.", "contexts": ["Query decomposition: split multi-part questions into sub-queries, run retrieval for each, merge contexts before passing to the LLM."], "ground_truth": "Complex queries are split into sub-queries, each retrieved independently, then merged."}

{"question": "What languages does the SDK support?", 
 "answer": "The SDK supports Python, JavaScript, Go, and Java.", "contexts": ["Official SDKs available for: Python, JavaScript/TypeScript, Go, and Java."], "ground_truth": "Python, JavaScript/TypeScript, Go, and Java SDKs are available."}

{"question": "How is faithfulness different from factual accuracy?", 
 "answer": "Faithfulness checks if the answer is supported by the retrieved context. Factual accuracy checks if the answer is true in the real world. An answer can be faithful but factually wrong if the context itself is wrong.", "contexts": ["Faithfulness measures grounding in retrieved context, not real-world truth.", "A faithful answer only claims what the context supports — it may still be factually wrong if the source is incorrect."], "ground_truth": "Faithfulness = grounded in context; accuracy = true in the world. Faithful answers can still be factually wrong."}

{"question": "What happens when I delete my account?", 
 "answer": "All your data is permanently deleted within 30 days and cannot be recovered.", "contexts": ["Account deletion: all user data is permanently purged within 30 days. This action is irreversible."], "ground_truth": "Data is permanently deleted within 30 days with no recovery option."}

{"question": "How do I reduce hallucinations in my RAG pipeline?", 
 "answer": "Use citation constraints in the system prompt, lower the LLM temperature, and add a post-generation NLI verification step.", "contexts": ["To reduce hallucinations: instruct the model to only use provided context, lower temperature, consider post-generation verification."], "ground_truth": "Citation constraints, lower temperature, and post-generation NLI verification reduce hallucinations."}

{"question": "What is chunk overlap and why does it matter?", 
 "answer": "Chunk overlap repeats a portion of text between adjacent chunks so that information near chunk boundaries is not lost during retrieval.", "contexts": ["Chunk overlap: repeating 10–20% of text between adjacent chunks prevents information loss at boundaries."], "ground_truth": "Overlap repeats boundary text between chunks to avoid missing information at split points."}

{"question": "How do I export my data?", 
 "answer": "Go to Settings → Data → Export and select CSV or JSON format. The export will be emailed to you.", "contexts": ["Data export: Settings → Data → Export. Choose CSV or JSON. A download link will be sent to your registered email."], "ground_truth": "Export data via Settings > Data > Export in CSV or JSON; link sent by email."}

{"question": "What vector databases are supported?", 
 "answer": "The platform integrates with Pinecone, Weaviate, Qdrant, and pgvector.", "contexts": ["Supported vector databases: Pinecone, Weaviate, Qdrant, and pgvector (PostgreSQL extension)."], "ground_truth": "Pinecone, Weaviate, Qdrant, and pgvector are supported."}

{"question": "How does re-ranking improve RAG performance?", 
 "answer": "Re-ranking applies a cross-encoder model after initial retrieval to reorder chunks by true relevance, improving context precision before passing to the LLM.", "contexts": ["Re-ranking: after bi-encoder retrieval, a cross-encoder scores each chunk against the query and reorders them. This improves precision at the cost of added latency."], "ground_truth": "Re-ranking reorders retrieved chunks by relevance using a cross-encoder, improving context precision."}

{"question": "What is the end-to-end RAG pipeline flow?", 
 "answer": "Query → embed query → vector search → retrieve top-k chunks → re-rank → build prompt with context → LLM generates answer.", "contexts": ["RAG pipeline: embed the query, search the vector store for top-k chunks, optionally re-rank, inject chunks into the prompt, generate answer with LLM."], "ground_truth": "Embed query → vector search → top-k retrieval → re-rank → prompt injection → LLM generation."}